# Widefield imaging: trial-averaged dF/F

This notebook averages the imaging frames across the trials of one recording session and turns the
result into **dF/F** — how much the signal changes from the pre-stimulus baseline.

The averaging is built to be **exact and safe to run in parallel**: splitting the trials into any
number of pieces (on your laptop, or across a cluster) gives the same result, bit for bit. The
short reason: each trial is added up on its own, whole camera frames add exactly, and the division
that makes the average — and the dF/F — happen once at the very end. See `docs/imaging_plan.md` for
the full explanation.

**You only need to edit one cell** — the session name in step 1 (and, optionally, the condition
column in step 5). Everything else runs from there.


## 1. Load your session and pick the imaging run

Change `SESSION` to your session folder name (the same name you use in your other analysis
notebooks). A session can contain several runs; we keep only the ones that actually recorded 1P
camera frames.


In [ ]:
from piepy.tasks.sensory.visual.visualSession import VisualSession

# EDIT: your session folder name.
SESSION = "260713_KC153__1P_KC"

sess = VisualSession(SESSION)   # load_flag=True reuses a previous parse instead

# Keep the runs that recorded 1P camera frames (run.paths.onepcam points to their image folder).
imaging_runs = [r for r in sess.runs if r.paths.onepcam is not None]
print(f"{len(imaging_runs)} run(s) with 1P imaging, out of {len(sess.runs)} total")

run = imaging_runs[0]            # EDIT: choose which imaging run to analyse
print("image folder:", run.paths.onepcam)


In [ ]:
for r in imaging_runs:
    r.analyze_run()

## 2. Look at the trial table

Every trial row carries `onepcam_frame_ids = [first_frame, last_frame]`: the camera frame numbers
recorded while the stimulus was on. Trials with no frames are skipped automatically.

(If `run.data.data` is empty, run `sess.analyze()` once first — building the session normally fills
it in.)


In [ ]:
trials = run.data.data
trials.select(["trial_no", "onepcam_frame_ids"]).head(10)


## 3. Frame timing

To turn your baseline/response windows (in milliseconds) into a number of frames, the pipeline needs
the average time between camera frames. It is read from the camera log in the image folder.


In [ ]:
from piepy.imaging.widefield import run_frame_period_ms

frame_t = run_frame_period_ms(run.paths.onepcam, timestamp_precision=0.001)
print(f"mean frame period: {frame_t:.3f} ms  (~{frame_t:.1f} Hz)")


## 4. Average the trials → dF/F (the one-liner)

`widefield_from_run` opens the frames, builds one fixed-length window per trial (with `pre_t`
milliseconds of baseline *before* the stimulus), averages the trials, and returns dF/F.

The result is a dictionary `{condition: movie}`. With no conditions there is a single entry under
the key `None`. Each movie has shape `(frames, height, width)`.


In [ ]:
from piepy.imaging.widefield import widefield_from_run

results = widefield_from_run(
    run,
    conditions=None,   # average all trials together (step 5 splits by stimulus)
    pre_t=200,         # ms of baseline before the stimulus (used as the dF/F reference)
    post_t=0,          # ms to keep after the stimulus window
    downsample=1,      # set e.g. 2 to halve height and width
)

movie = results[None]
print("movie shape (frames, H, W):", movie.shape)


## 5. Average separately per stimulus condition

Pass `conditions` — one column name, or a list of them — to get one averaged movie per condition
(e.g. one per contrast). This is the same idea as grouping a trial table before averaging.


In [ ]:
# EDIT: the column(s) that define your conditions, e.g. "contrast" or ["contrast", "opto"].
CONDITION = "contrast"

by_cond = widefield_from_run(run, conditions=CONDITION, pre_t=100)
for key, mov in by_cond.items():
    print(f"{CONDITION} = {key!r}:  {mov.shape}")


## 6. The averaging is exact when split into pieces

This is the whole point of the pipeline. `n_pieces` splits the trials into that many groups — which
is what lets the work run in parallel — and it does **not** change the result at all.


In [ ]:
import numpy as np

one   = widefield_from_run(run, conditions=None, pre_t=100, n_pieces=1)
eight = widefield_from_run(run, conditions=None, pre_t=100, n_pieces=8)

assert np.array_equal(one[None], eight[None])
print("1 piece and 8 pieces give the identical movie ✔")


### Running on a cluster

For real speed-ups, run the pieces as separate processes or as separate cluster jobs. Because each
piece only produces a running total that is added up at the end, the result stays identical.

* On one machine: pass `executor=ProcessExecutor(n_workers)`.
* On a cluster: run one job per group of trials, save each group's running total with `partial_sum`,
  and add them together with `combine` in a final job.

Note: sending the image stack to worker processes requires it to be picklable. If your stack keeps
open file handles, prefer the one-job-per-group pattern above. The line below is left commented out.


In [ ]:
# from piepy.imaging.executor import ProcessExecutor
# results = widefield_from_run(run, pre_t=100, n_pieces=8, executor=ProcessExecutor(4))


## 7. The same thing, step by step

If you want the intermediate results (to line frames up with events, pick ROIs, and so on), here are
the exact steps the one-liner runs.


In [ ]:
from piepy.imaging.onep.stacks import load_stack
from piepy.imaging.windows import frame_windows
from piepy.imaging.average import trial_average, dff

import os
import tifffile as tf
from piepy.imaging.widefield import save_averages, to_display_uint16

for r_run in imaging_runs:
    stack = load_stack(r_run.paths.onepcam, nchannels=1)
    trials = r_run.data.data
    # 1) one fixed-length frame window per trial (the first `pre` frames are the baseline)
    windows = frame_windows(trials, mode="onepcam", pre_t=300, post_t=100, frame_t=frame_t)
    print("trials kept:", windows.n, "| frames per trial:", windows.count, "| baseline frames:", windows.pre)

    # 2) average the trial frames (this is the plain mean, not yet dF/F)
    mean = trial_average(stack, windows)[None]

    # 3) dF/F against the baseline frames — done once, on the finished average
    movie = dff(mean, windows.pre)
    
    out_dir = r_run.paths.save[0]
    paths = save_averages({None:movie}, out_dir)          # float32, for analysis

    print("saved:", out_dir)
    # tf.imwrite(os.path.join(out_dir, "avg.tif"), np.asarray(movie, dtype=np.float32))
    # optional: a display copy (values rescaled for viewing only)
    tf.imwrite(os.path.join(out_dir, "avg_display.tif"), to_display_uint16(movie))

In [ ]:
results.keys()

## 8. Look at the result

One frame shortly after the stimulus, and the time course of a small central region.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# frame ~200 ms after the stimulus (the first `windows.pre` frames are the baseline)
peak = min(windows.pre + int(round(200 / frame_t)), movie.shape[0] - 1)

fig, (a0, a1) = plt.subplots(1, 2, figsize=(9, 4))

a0.imshow(movie[peak], cmap="magma")
a0.set_title(f"dF/F at frame {peak} (~200 ms)")
a0.axis("off")

# average dF/F inside a small central box, over time
h, w = movie.shape[1:]
box = movie[:, h // 2 - 10:h // 2 + 10, w // 2 - 10:w // 2 + 10].mean(axis=(1, 2))
t_ms = (np.arange(movie.shape[0]) - windows.pre) * frame_t   # time from stimulus onset

a1.plot(t_ms, box)
a1.axvline(0, color="k", ls=":")     # stimulus onset
a1.set_xlabel("time from stimulus (ms)")
a1.set_ylabel("dF/F")
a1.set_title("central region")
plt.tight_layout()


## 9. Save

Save the analysis movies as float32 tiffs — this keeps the real dF/F values. For a quick look in
ImageJ and similar, also save a display copy stretched to the full 16-bit range. **Use the display
copy for viewing only**, never for measurements.


In [ ]:
import os
import tifffile as tf
from piepy.imaging.widefield import save_averages, to_display_uint16

out_dir = run.paths.save[0]
paths = save_averages(results, out_dir)          # float32, for analysis
print("saved:", paths)

# optional: a display copy (values rescaled for viewing only)
tf.imwrite(os.path.join(out_dir, "avg_display.tif"), to_display_uint16(results[None]))
